# This code annotates the videos -- it will take a long time (hours) to run

In [ ]:
# Point this to your cropped video folder that contains only crops
# Make sure that each crop is named with the animal number
video_folder = r""

# Set this to True if you want to create videos with the annotations overlaid
# Set to False to save space
create_annotation_videos = True

In [ ]:
import os
import torch
import deeplabcut
import shutil
from pathlib import Path

# Force PyTorch to use CPU if the GPU isn't working
# os.environ["CUDA_VISIBLE_DEVICES"] = ""

# Check GPU availability
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: PyTorch cannot detect a CUDA GPU.")

# Path to your DLC project config.yaml
config_path = r""

# Grab all supported video files in the folder
video_list = [
    os.path.join(video_folder, f)
    for f in os.listdir(video_folder)
    if f.lower().endswith(('.mp4', '.avi', '.mov', '.mkv'))
]

# Optional: print device being used
# print("Using device:", "cuda" if torch.cuda.is_available() else "cpu")

# Analyze videos
deeplabcut.analyze_videos(config_path, video_list, save_as_csv=True)

# Create annotated videos in a new subfolder
if create_annotation_videos:
    annotated_folder = Path(video_folder) / "Annotated_Videos"
    annotated_folder.mkdir(exist_ok=True)

    deeplabcut.create_labeled_video(
        config_path,
        video_list,
        draw_skeleton=False,
        trailpoints=0
    )

    for video in video_list:
        for labeled in Path(video).parent.glob(
            f"{Path(video).stem}*labeled.mp4"
        ):
            shutil.move(
                str(labeled),
                str(annotated_folder / labeled.name)
            )
